# Deploying: saving, serving, and making it smaller

What happens after fit() returns — the .keras format, inference-only export, and the two ways to make a model cheaper to run.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 6 — The Universal Workflow of Machine Learning](../../../course-web-slides/ch06/index.html) &nbsp;·&nbsp; **Section:** 03 — Deploy the model

---

## A model to deploy

In [ ]:
import keras
from keras import layers
from keras.datasets import mnist
import numpy as np

(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

keras.utils.set_random_seed(0)
model = keras.Sequential([
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(x, y, epochs=3, batch_size=128, verbose=0)
base_acc = model.evaluate(xt, yt, verbose=0)[1]
print(f"accuracy: {base_acc:.4f}   parameters: {model.count_params():,}")

## Saving, and what is in the file

In [ ]:
import os, zipfile

model.save("mnist.keras")
size = os.path.getsize("mnist.keras")
print(f"mnist.keras: {size/1e6:.2f} MB")

with zipfile.ZipFile("mnist.keras") as z:
    for n in z.namelist():
        print(" ", n, f"{z.getinfo(n).file_size/1e6:.2f} MB")

Expected output:

```
mnist.keras: 1.2x MB
  metadata.json 0.00 MB
  config.json 0.00 MB
  model.weights.h5 1.2x MB
```

A zip of three things: the architecture as JSON, the weights, and a little metadata. **Backend-independent** — chapter 3's notebook 02 saved under one backend and loaded under another.

## Preprocessing belongs inside the exported model

The most common serving bug is not a broken model — it is preprocessing that differs by a fraction between the training script and the service. Put it in the graph and the question cannot arise.

In [ ]:
inputs = keras.Input(shape=(28, 28), dtype="uint8", name="image")
x_ = keras.layers.Rescaling(1./255)(keras.ops.cast(inputs, "float32"))
x_ = keras.layers.Reshape((784,))(x_)
outputs = model(x_)
servable = keras.Model(inputs, outputs, name="mnist_servable")

# It now takes exactly what the caller has: raw uint8 images.
raw = mnist.load_data()[1][0][:4]
print("input dtype:", raw.dtype, " shape:", raw.shape)
print("predictions:", servable.predict(raw, verbose=0).argmax(axis=1))
servable.save("mnist_servable.keras")

## Making it smaller: int8 quantization

Chapter 18 explains the arithmetic. Here is what it costs and what it buys, measured.

In [ ]:
import copy, time

q = keras.saving.load_model("mnist.keras")
q.quantize("int8")
q_acc = q.evaluate(xt, yt, verbose=0)[1]
q.save("mnist_int8.keras")

fp = os.path.getsize("mnist.keras") / 1e6
qs = os.path.getsize("mnist_int8.keras") / 1e6
print(f"float32: {fp:.2f} MB   accuracy {base_acc:.4f}")
print(f"int8:    {qs:.2f} MB   accuracy {q_acc:.4f}")
print(f"size: {fp/qs:.1f}x smaller   accuracy cost: {base_acc-q_acc:+.4f}")

Expected output:

```
float32: 1.2x MB   accuracy 0.97xx
int8:    0.3x MB   accuracy 0.97xx
size: ~4x smaller   accuracy cost: -0.00xx
```

> **Note** — Measure the accuracy cost on **your** data, every time. It is usually negligible and it is not guaranteed to be — and quantizing is a one-way operation on the model object, so keep the float32 file.

## Measuring inference cost honestly

In [ ]:
def bench(m, data, n=5, warmup=2):
    for _ in range(warmup):
        m.predict(data, verbose=0)
    t0 = time.perf_counter()
    for _ in range(n):
        m.predict(data, verbose=0)
    return (time.perf_counter() - t0) / n

batch = xt[:512]
print(f"float32: {bench(model, batch)*1000:.1f} ms / 512 samples")
print(f"int8:    {bench(q, batch)*1000:.1f} ms / 512 samples")

**Warm up first.** The first call compiles; timing it measures the compiler. Chapter 16's generation notebook makes the same mistake deliberately, and it costs two orders of magnitude there.

## What still has to be built around this

The model is the small part. A deployment also needs:

- **Input validation** — the shape and dtype the model expects, enforced at the boundary rather than assumed.
- **Monitoring** — not just errors, but the distribution of inputs. Chapter 19's point about distribution shift starts counting from the day you deploy.
- **A rollback** — models are data, and a bad model ships as easily as a good one.
- **A held-out set you have not touched**, so the number you quote is one you can defend.

---

## What to take away

- `.keras` is a zip of config plus weights, and it is backend-independent.
- Put preprocessing **inside** the exported model — it removes the most common serving bug.
- int8 quantization is roughly 4× smaller; measure the accuracy cost on your own data.
- Warm up before benchmarking, or you are timing the compiler.